# NRCH two-loop flow — Julia + Makie

Recreates the 3D flow plot and the `bu = 0` isosurface from `NRCH_twoloop.ipynb`.
Loads the integrals precomputed by the Python notebook from `data/samples.npz`.

In [ ]:
# Run once to install dependencies.
using Pkg
# Pkg.add(["NPZ", "Interpolations", "Meshing", "GeometryBasics", "LaTeXStrings"])
# Plus whichever Makie backend you want (uncomment in the imports cell):
# Pkg.add("GLMakie")              # A: popup window & B: inline PNG
# Pkg.add(["WGLMakie", "Bonito"]) # C: WGL inline interactive
# Pkg.add("CairoMakie")           # D: static inline
# Pkg.add("LaTeXStrings")

In [ ]:
using NPZ
using Interpolations
using Meshing
using GeometryBasics
using Random
using LaTeXStrings

# =================================================================
# PICK ONE BACKEND — uncomment exactly one block below.
# RESTART THE KERNEL before re-running after a change; Makie backends
# can't cleanly swap in a live session.
# Each block defines `show_fig(fig)` — figure cells call that.
# =================================================================

In [ ]:
# # --- A: GLMakie popup (interactive OS window) ---  [DEFAULT, verified]

using GLMakie
GLMakie.activate!()
Makie.inline!(false)
# show_fig(fig) = display(GLMakie.Screen(), fig)
show_fig(fig) = display(fig)

In [ ]:
data = npzread("data/samples.npz")

b_samples = data["b_vals"]
I_ff   = data["I_ff_real"]   .+ im .* data["I_ff_imag"]
I_dff  = data["I_dff_real"]  .+ im .* data["I_dff_imag"]
I_ddff = data["I_ddff_real"] .+ im .* data["I_ddff_imag"]

length(b_samples), extrema(b_samples)

In [ ]:
B(b) = (1 - im*b) / (1 + im*b)

g_samples = @. (1 + 8 * (1 + B(b_samples)) * (I_dff + 2 * B(b_samples) * I_ddff)) / (1 + im * b_samples)

# Cubic spline interpolation of the real and imaginary parts on the uniform grid.
g_re_itp = cubic_spline_interpolation(range(b_samples[1], b_samples[end]; length=length(b_samples)), real.(g_samples))
g_im_itp = cubic_spline_interpolation(range(b_samples[1], b_samples[end]; length=length(b_samples)), imag.(g_samples))

gs(b) = g_re_itp(b) + im * g_im_itp(b)

gs(0.0)  # should be ~1/3

In [ ]:
d = 3
ϵ = 4 - d

h(a, b) = (1 + b*a) * real(gs(b)) - (a - b) * imag(gs(b))

# βu(u, a, b, c) = -ϵ*u + 2*u^2/(b^2 + 1) * ((4*b^2 + 5) - a^2 + 2*b*a)
βu(u, a, b, c) = -ϵ*u + 10*u^2 * (1 - 1/5 * (a-b)^2 / (1+b^2))
βa(u, a, b, c) = +2*u*(a - b) * (a^2 + 1) / (b^2 + 1)
βb(u, a, b, c) = +2*u^2 * (b - a) * (1 + h(a, b))
βc(u, a, b, c) = -4 * u * (c - a)

# Flow on the RG "upward" direction (negative beta).
v(u, a, b, c) = (-βu(u,a,b,c), -βa(u,a,b,c), -βb(u,a,b,c), -βc(u,a,b,c))

eq(u, a, b) = βu(u, a, b, 0.0)

In [ ]:
function simulate(v, u0, a0, b0, c0; dt=0.02, nsteps=1000)
    traj = zeros(nsteps, 4)
    traj[1, :] = [u0, a0, b0, c0]
    for i in 2:nsteps
        u, a, b, c = traj[i-1, :]
        du, da, db, dc = v(u, a, b, c)
        traj[i, :] = [u + dt*du, a + dt*da, b + dt*db, c + dt*dc]
    end
    return traj
end

function get_init(N, ur, ar, br, cr; rng=Random.default_rng())
    inits = Vector{NTuple{4,Float64}}()
    for _ in 1:N
        u0 = rand(rng) * ur
        a0 = (2*rand(rng) - 1) * ar
        b0 = rand(rng) * br
        c0 = a0
        # c0 = 0
        push!(inits, (u0, a0, b0, c0))
    end
    return inits
end

In [ ]:
function plot_flow3D!(ax, v, N, ur, ar, br, cr;
                       nsteps=1000, cmap=:plasma, crange=(-10.0, 10.0),
                       seed=nothing, linewidth=3, subsample=5, ms=10)
    rng = seed === nothing ? Random.default_rng() : Random.MersenneTwister(seed)
    inits = get_init(N, ur, ar, br, cr; rng=rng)
    trajs = [simulate(v, u0, a0, b0, c0; nsteps=nsteps) for (u0, a0, b0, c0) in inits]

    cmin, cmax = Float32(crange[1]), Float32(crange[2])

    # Build one big NaN-separated strip so we get a single lines! draw call.
    # NOTE: NaN goes only in x/y/z (that's what breaks the line); the color array
    # uses a finite sentinel because NaN in per-vertex color can confuse shaders.
    segs = [traj[1:subsample:end, :] for traj in trajs]
    npts = sum(size(s, 1) for s in segs) + length(segs)  # +1 sep per traj

    xs = Vector{Float32}(undef, npts)
    ys = Vector{Float32}(undef, npts)
    zs = Vector{Float32}(undef, npts)
    cs = Vector{Float32}(undef, npts)

    i = 1
    for s in segs
        n = size(s, 1)
        @inbounds for k in 1:n
            xs[i] = s[k, 2]
            ys[i] = s[k, 3]
            zs[i] = s[k, 1]
            cs[i] = clamp(Float32(s[k, 4]), cmin, cmax)
            # cs[i] = @. abs( (s[k, 4] - s[k, 2]) / s[k, 2])
            i += 1
        end
        # Strip separator: NaN in position only.
        xs[i] = NaN32; ys[i] = NaN32; zs[i] = NaN32; cs[i] = cmin
        i += 1
    end

    start_x = Float32[t[1, 2]     for t in trajs]
    start_y = Float32[t[1, 3]     for t in trajs]
    start_z = Float32[t[1, 1]     for t in trajs]
    end_x   = Float32[t[end, 2]   for t in trajs]
    end_y   = Float32[t[end, 3]   for t in trajs]
    end_z   = Float32[t[end, 1]   for t in trajs]
    end_c   = Float32[clamp(Float32(t[end, 4]), cmin, cmax) for t in trajs]

    lines!(ax, xs, ys, zs;
           color=cs, colormap=cmap, colorrange=crange, linewidth=linewidth,fxaa=true
           )
    scatter!(ax, start_x, start_y, start_z;
             color=:red, markersize=ms)
    scatter!(ax, end_x, end_y, end_z;
             color=end_c, colormap=cmap, colorrange=crange, markersize=ms)
    return trajs
end

In [ ]:
set_theme!(theme_latexfonts())

ur = .2
ar = 1.5
br = ar
cr = ar

function get_init(N, ur, ar, br, cr; rng=Random.default_rng())
    return [
        (0.01,  +0.5,     0,    +0.5,),
        (0.20,  +0,     1.5,     +0, ),
        (0.20,  +0,     0.5,     +0, ),
        (0.20,  -1.5,     1,    -1.5,),
        (0.20,  -1.5,     1.5,  -1.5,),
        (0.01,  -1.5,     1,    -1.5,),
        (0.05,  -1.5,     .5,   -1.5,),
        (0.05,  +1.5,     .2,   +1.5,),
        (0.20,  +1.2,     1.,   +1.2,),
    ]
end
N = get_init(0,0,0,0,0)

l = 50
crange=(-l,l)
cmap = :hot
# cmap = :oslo
color = :seagreen3
N = 8

s = 15
fs = 50
ls = 50
ms = 30
lw = 10

fig = Figure(size=(150*s, 100*s); fontsize = 30)
ax  = Axis3(fig[1, 1]; 
    xlabel=L"a", ylabel=L"b", zlabel=L"\lambda", viewmode = :fit,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs,
    zlabeloffset = 100, 
    # xticklabelsize = ls, yticklabelsize = ls, zticklabelsize = ls,
    # titlesize = 24
    )

trajs = plot_flow3D!(ax, v, N, ur, ar, br, cr;
                     nsteps=3000, cmap=cmap, seed=nothing, linewidth=lw, subsample=5, crange=crange, ms=ms)

u_grid = collect(range(0.01, ur;    length=30))
a_grid = collect(range(-0.8*br, br; length=60))
b_grid = collect(range(-0.1*br, br;     length=60))

V = [eq(u, a, b) for a in a_grid, b in b_grid, u in u_grid]

pts, tris = isosurface(V, MarchingCubes(iso=0.0), a_grid, b_grid, u_grid)
verts = [Point3f(p...) for p in pts]
faces = [TriangleFace{Int}(f...) for f in tris]
surface_mesh = GeometryBasics.Mesh(verts, faces)

mesh!(ax, surface_mesh; color=(color, .8), transparency=true)

Colorbar(fig[1, 2]; colormap=cmap, limits=crange, label=L"c", width=50, height = 400, tellheight = false,
    valign = :center, ticklabelsize=fs, labelsize=fs)
show_fig(fig)

# Plot flow in abc

In [ ]:
yellow      = colorant"#fff800"
pink        = colorant"#ffdcfe"
blue        = colorant"#a3f8ff"
green       = colorant"#c8ffce"
turquise    = colorant"#41e3c0"
red         = RGBf(0.8,0.2,0.2)


function plot_flowabc!(ax, v, N, ur, ar, br, cr;
                       nsteps=1000, cmap=:plasma, crange=(-10.0, 10.0),
                       seed=nothing, linewidth=3, subsample=5, ms=10,
                       arrow_col=nothing)
    rng = seed === nothing ? Random.default_rng() : Random.MersenneTwister(seed)
    inits = get_init(N, ur, ar, br, cr; rng=rng)
    trajs = [simulate(v, u0, a0, b0, c0; nsteps=nsteps) for (u0, a0, b0, c0) in inits]

    cmin, cmax = Float32(crange[1]), Float32(crange[2])

    segs = [traj[1:subsample:end, :] for traj in trajs]
    npts = sum(size(s, 1) for s in segs) + length(segs)  # +1 sep per traj

    xs = Vector{Float32}(undef, npts)
    ys = Vector{Float32}(undef, npts)
    zs = Vector{Float32}(undef, npts)
    cs = Vector{Float32}(undef, npts)

    i = 1
    for s in segs
        n = size(s, 1)
        @inbounds for k in 1:n
            xs[i] = s[k, 2]
            ys[i] = s[k, 3]
            zs[i] = s[k, 4]
            i += 1
        end
        # Strip separator: NaN in position only.
        xs[i] = NaN32; ys[i] = NaN32; zs[i] = NaN32; cs[i] = cmin
        i += 1
    end

    start_x = Float32[t[1, 2]     for t in trajs]
    start_y = Float32[t[1, 3]     for t in trajs]
    start_z = Float32[t[1, 4]     for t in trajs]
    end_x   = Float32[t[end, 2]   for t in trajs]
    end_y   = Float32[t[end, 3]   for t in trajs]
    end_z   = Float32[t[end, 4]   for t in trajs]

    lines!(ax, xs, ys, zs; color=:black, linewidth=linewidth,fxaa=true)
    scatter!(ax, start_x, start_y, start_z;color=:blue, markersize=ms)

    # Cone-head parameters for the end-of-line markers.
    head_len = 0.18
    head_radius = 0.08
    head_shift = -0.1
    n_sides = 10

    if arrow_col === nothing; head_colors = fill(:black, length(segs))
    else; head_colors = collect(arrow_col)
    end

    for (idx, s) in enumerate(segs)
        if size(s, 1) >= 2
            p_prev = (Float64(s[end-1, 2]), Float64(s[end-1, 3]), Float64(s[end-1, 4]))
            p_curr = (Float64(s[end, 2]), Float64(s[end, 3]), Float64(s[end, 4]))
            v = (p_curr[1] - p_prev[1], p_curr[2] - p_prev[2], p_curr[3] - p_prev[3])
            vnorm = sqrt(v[1]^2 + v[2]^2 + v[3]^2)
            if vnorm > 1e-8
                u = (v[1] / vnorm, v[2] / vnorm, v[3] / vnorm)
                ref = abs(u[3]) > 0.9 ? (1.0, 0.0, 0.0) : (0.0, 0.0, 1.0)
                n = (
                    u[2] * ref[3] - u[3] * ref[2],
                    u[3] * ref[1] - u[1] * ref[3],
                    u[1] * ref[2] - u[2] * ref[1],
                )
                nlen = sqrt(n[1]^2 + n[2]^2 + n[3]^2)
                if nlen > 1e-8
                    n = (n[1] / nlen, n[2] / nlen, n[3] / nlen)
                    b = (
                        u[2] * n[3] - u[3] * n[2],
                        u[3] * n[1] - u[1] * n[3],
                        u[1] * n[2] - u[2] * n[1],
                    )
                    tip = ntuple(i -> p_curr[i] - head_shift * u[i], 3)
                    base = ntuple(i -> tip[i] - head_len * u[i], 3)
                    ring = []
                    for θ in range(0, 2π; length=n_sides)
                        c = cos(θ)
                        s = sin(θ)
                        push!(ring, ntuple(i -> base[i] + head_radius * (c * n[i] + s * b[i]), 3))
                    end
                    for i in 1:length(ring)
                        j = i % length(ring) + 1
                        poly!(ax, [
                            Point3f(tip[1], tip[2], tip[3]),
                            Point3f(ring[i][1], ring[i][2], ring[i][3]),
                            Point3f(ring[j][1], ring[j][2], ring[j][3]),
                        ]; color=head_colors[idx], strokewidth=0)
                    end
                end
            end
        end
    end
    return trajs
end

In [ ]:
set_theme!(theme_latexfonts())

ur = .2
ar = 1.2
br = ar
cr = ar

function get_init(N, ur, ar, br, cr; rng=Random.default_rng())
    return [
        (0.1,  +1,     0,    +.6,),
        (0.1,  -1,     0,    -.6,),
        (0.1,  -1,     0,    +.6,),
        (0.1,  +1,     0,    -.6,),
        (0.1,  1,     -1,    .475,),
    ]
end
N = get_init(0,0,0,0,0)

N = 8
s = 15
fs = 50
ls = 50
ms = 30
lw = 10

fig = Figure(size=(150*s, 100*s); fontsize = 30)
ax  = Axis3(fig[1, 1]; 
    xlabel=L"\alpha_1/u", ylabel=L"\beta_0/K", zlabel=L"\alpha_0/r", viewmode = :fit,
    xlabelsize = fs, ylabelsize = fs, zlabelsize = fs,
    xlabeloffset = 50,
    ylabeloffset = 50,
    zlabeloffset = 70,
    )

xlims!(ax, (-ar, ar))
ylims!(ax, (-ar, ar))
zlims!(ax, (-ar, ar))

# Make the scaling equal on all axes.
ax.aspect = :data

vals = range(-ar, ar; length=200)
lines!(ax, vals, vals, vals; color=red, linewidth=10)

# Plot the flow trajectories.
trajs = plot_flowabc!(ax, v, N, ur, ar, br, cr; 
    nsteps=500, linewidth=lw, ms=ms, arrow_col=(blue, green, green, blue, pink))

show_fig(fig)

In [ ]:
save("flow3D4.png", ax.scene; px_per_unit=2)